### Load in the Model (DeepSeek-R1-Distill-Llama-8B)

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B")
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B", device_map="cuda", dtype=torch.bfloat16)
print(model.device)
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

c:\Users\Faruk\miniconda3\envs\algoverse\Lib\site-packages\transformers\modeling_utils.py:5198: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  _ = torch.empty(int(byte_count // 2), dtype=torch.float16, device=device, requires_grad=False)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

cuda:0
Greetings! I'm DeepSeek-R1, an artificial intelligence assistant created by DeepSeek. I'm at your service and would be delighted to assist you with any inquiries or tasks you may have.
</think>


### Loading the Datasets

In [3]:
import pandas as pd

# Load in the Factual Datasets
F0_train, F0_test = pd.read_csv("../dataset/F0_train.csv")[["statement", "label"]], pd.read_csv("../dataset/F0_test.csv")[["statement", "label"]]
F1_train, F1_test = pd.read_csv("../dataset/F1_train.csv")[["statement", "label"]], pd.read_csv("../dataset/F1_test.csv")[["statement", "label"]]
F2_train, F2_test = pd.read_csv("../dataset/F2_train.csv"), pd.read_csv("../dataset/F2_test.csv")
F3_train, F3_test = pd.read_csv("../dataset/F3_train.csv"), pd.read_csv("../dataset/F3_test.csv")
F4_train, F4_test = pd.read_csv("../dataset/F4_train.csv"), pd.read_csv("../dataset/F4_test.csv")
F5_train, F5_test = pd.read_csv("../dataset/F5_train.csv"), pd.read_csv("../dataset/F5_test.csv")

# Load in the Arithmatic Statements
A1_train, A1_test = pd.read_csv("../dataset/A1_train.csv"), pd.read_csv("../dataset/A1_test.csv")
A2_train, A2_test = pd.read_csv("../dataset/A2_train.csv"), pd.read_csv("../dataset/A2_test.csv")
A3_train, A3_test = pd.read_csv("../dataset/A3_train.csv"), pd.read_csv("../dataset/A3_test.csv")


In [4]:
datasets = {
    "F0_train": F0_train, "F0_test": F0_test,
    "F1_train": F1_train, "F1_test": F1_test,
    "F2_train": F2_train, "F2_test": F2_test,
    "F3_train": F3_train, "F3_test": F3_test,
    "F4_train": F4_train, "F4_test": F4_test,
    "F5_train": F5_train, "F5_test": F5_test,
    "A1_train": A1_train, "A1_test": A1_test,
    "A2_train": A2_train, "A2_test": A2_test,
    "A3_train": A3_train, "A3_test": A3_test,
}

for name, df in datasets.items():
    print(f"{name}: {len(df)}")


F0_train: 1194
F0_test: 512
F1_train: 1194
F1_test: 512
F2_train: 1194
F2_test: 512
F3_train: 1398
F3_test: 600
F4_train: 1394
F4_test: 598
F5_train: 1383
F5_test: 593
A1_train: 700
A1_test: 300
A2_train: 700
A2_test: 300
A3_train: 700
A3_test: 300


### Funtions to Generate Activations

In [5]:
import re
import torch
from tqdm import tqdm
from pathlib import Path
from ast import literal_eval

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# DeepSeek-R1-Distill always reasons inside <think>...</think> on its own — we don't need
# to force that structure via instructions, just guide *what* the reasoning should cover.
# Keep this loose: an over-rigid "one line per city, no summary" instruction fights the
# model's native reasoning style (that fought Llama-3.1-8B into unreliable arithmetic).
#
# A literal marker phrase ("FINAL VERDICT:") did NOT work: across every sampled trace,
# the model ignored the instruction to place that exact marker inside <think> and instead
# closed </think> with its own natural conclusion sentence, then invented its own
# "Final Verdict"-style heading in the *answer* section.
#
# We still ask it to reserve True/False for its conclusion (reduces incidental noise),
# but we don't rely on that alone: a forward keyword search fails on its own too, because
# every trace OPENS with "...trying to figure out whether the statement '...' is true or
# false", which is a false-positive hit on the very first sentence, long before any real
# reasoning. So we anchor backward from </think> instead, restrict the search to only the
# last few lines before it (the "commit region"), and match a wider set of phrasings
# DeepSeek actually uses to commit ("is incorrect", "does not hold", "does not match",
# etc.), not just bare true/false.
COT_INSTRUCTIONS = (
    "Work through the statement above before answering. State the facts you need — for "
    "each city named, where it actually is; for each arithmetic sub-expression, its "
    "actual value — then explicitly compute or count what the statement claims, and "
    "compare.\n\n"
    "Do not use the words True, False, correct, incorrect, accurate, or inaccurate while "
    "reasoning. Reserve them for your conclusion only.\n\n"
    "End your reasoning with exactly one line, either:\n"
    "The statement is true.\n"
    "or\n"
    "The statement is false.\n"
    "Then write </think>. After </think>, on its own line, write exactly one of:\n"
    "Answer: Yes\n"
    "or\n"
    "Answer: No"
)




# Ordered roughly by how DeepSeek commits. All matched case-insensitively as whole words /
# phrases. Add any phrasing you see slip through during an audit_cuts() pass. Bare
# true/false go last so the more specific phrase forms win when both are present on a line.
#
# NOTE: earlier versions of this list also included bare-transition patterns like
# r"\btherefore,?\s+the\s+statement\b" and r"\bso\s+the\s+(?:answer|statement)\b".
# Those were a real bug: "Therefore, the statement" doesn't itself reveal anything —
# the actual reveal can come several words or even a sentence later ("...is saying 0
# are in Tanzania when in fact 1 is. That makes the entire statement false."). Matching
# on the bare connective cut way too early, discarding legitimate reasoning between the
# connective and the real reveal word. Every pattern below requires an actual reveal
# word (true/false/correct/incorrect/accurate/hold/match) to be present, so cutting
# only ever happens right before genuine information leaks, never before a mere
# transition phrase.
LEAD_IN = r"(?:is|to\s+be|seems(?:\s+to\s+be)?)"

VERDICT_PATTERNS = [
    rf"\b{LEAD_IN}\s+((?:in)?correct)\b",
    rf"\b{LEAD_IN}\s+(?:not\s+)?(true)\b",
    rf"\b{LEAD_IN}\s+(false)\b",
    rf"\b{LEAD_IN}\s+((?:in)?accurate)\b",
    r"\bdoes(?:n't| not)\s+(hold)\b",
    r"\bdoes(?:n't| not)\s+(match)\b",
    r"\b(?:does(?:n't| not)\s+)?(aligns?)\s+(?:perfectly\s+)?with\b",
    r"\bstatement\s+is\s+(?:therefore\s+)?(true|false|correct|incorrect)\b",
    r"\b((?:in)?correct)\b",   # bare fallback, mirrors true/false
    r"\b((?:in)?accurate)\b",  # bare fallback
    r"(true)\b",
    r"(false)\b",
]

VERDICT_RE = re.compile("|".join(VERDICT_PATTERNS), re.IGNORECASE)
ANCHOR_RE = re.compile(
    r"\b(?:therefore|thus|hence|in\s+summary|in\s+conclusion|to\s+conclude|overall)\b",
    re.IGNORECASE
)

# How many lines above </think> count as the "commit region". DeepSeek's conclusion is
# 1-3 lines; 4 gives margin without letting the opening restatement back into scope.
COMMIT_REGION_LINES = 4

# </think>'s exact token id sequence for this tokenizer, computed once. Used to anchor
# the backward search at the token-id level (robust to skip_special_tokens stripping it
# from decoded text, if it's registered as a special token).
THINK_CLOSE_IDS = tokenizer.encode("</think>", add_special_tokens=False)


def _find_think_close(ids, think_close_ids):
    """Token index where </think> starts, searching from the end since it's near there."""
    k = len(think_close_ids)
    for j in range(len(ids) - k, -1, -1):
        if ids[j:j + k] == think_close_ids:
            return j
    return None


def _char_to_token_idx(tokenizer, ids, upper_tok, target_char):
    lo, hi = 0, upper_tok
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if len(tokenizer.decode(ids[:mid], skip_special_tokens=True)) < target_char:
            lo = mid
        else:
            hi = mid - 1
    return lo

def _mask_parens(text):
    """Blank out the contents of any (...) spans (same length, so character
    offsets elsewhere stay valid) before searching for a verdict match.
    Targets parenthetical asides like "(which seems correct)" that comment
    on a sub-claim without being the actual final verdict, without the
    broader over-skipping problem the anchor-based approach had."""
    return re.sub(r"\([^()]*\)", lambda m: " " * len(m.group(0)), text)

def find_pre_verdict_cut(tokenizer, ids, think_close_ids=THINK_CLOSE_IDS):
    """
    ids: 1D list of token ids for the generated continuation only (no prompt).

    Returns a token index into `ids` to cut at (activation = last token of ids[:cut]),
    positioned just before the model commits to a verdict.

    Strategy:
      1. Find </think>. If absent (truncated generation), return len(ids).
      2. Look only at the last COMMIT_REGION_LINES lines before </think> — this keeps
         the opening restatement ("...is true or false") out of scope entirely.
      3. First verdict-phrase match in that region -> cut just before it.
      4. No match in region -> cut at the newline before </think> (best structural
         fallback; a sign VERDICT_PATTERNS needs to be widened further).
    """
    ids = list(ids)
    n = len(ids)
    if n == 0:
        return 0

    close_at = _find_think_close(ids, think_close_ids)
    if close_at is None:
        return n

    pre = tokenizer.decode(ids[:close_at], skip_special_tokens=True)

    pre_lines = pre.split("\n")
    region_start_char = len("\n".join(pre_lines[:-COMMIT_REGION_LINES]))
    if region_start_char > 0:
        region_start_char += 1  # step over the newline separating the region from the rest

    region_text = pre[region_start_char:]
    search_text = _mask_parens(region_text)
    # anchor_matches = list(ANCHOR_RE.finditer(region_text))
    # m_anchor = anchor_matches[-1] if anchor_matches else None

    # if m_anchor:
    #     m = VERDICT_RE.search(region_text[m_anchor.end():])
    #     if m is not None:
    #         target_char = region_start_char + m_anchor.end() + m.start(m.lastindex)
    #     else:
    #         m = VERDICT_RE.search(region_text)
    #         if m is not None:
    #             target_char = region_start_char + m.start(m.lastindex)
    # else:

    m = VERDICT_RE.search(search_text)

    if m is not None:
        target_char = region_start_char + m.start(m.lastindex)
        return _char_to_token_idx(tokenizer, ids, close_at, target_char)


    # Fallback: no known verdict phrasing found in the commit region. Cut at the
    # newline before </think> rather than leaking all the way up to it.
    last_nl = pre.rfind("\n")
    if last_nl == -1:
        return close_at
    return _char_to_token_idx(tokenizer, ids, close_at, last_nl)

def generate_activations(model, statements, layer, with_chat_template=True, batch_size=16):
    statements = list(statements)
    final_token_activations = []

    for i in range(0, len(statements), batch_size):
        statements_temp = statements[i: i+batch_size]
        if with_chat_template:
            messages = [[{"role": "user", "content": s}] for s in statements_temp]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to(model.device)
        else:
            inputs = tokenizer(statements_temp, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        final_token_activation = outputs.hidden_states[layer][:, -1, :]
        final_token_activations.append(final_token_activation.cpu())

    return torch.cat(final_token_activations, dim=0)


# With instructions and few shot
def generate_activations_cot(model, statements, layer, task, with_chat_template=True, batch_size=16, verbose=False, save_datasets=False):
    final_token_activations = []
    dataset_path = Path(f"../CoT_datasets/raw/{task}.csv")
    if not dataset_path.is_file():
        # Define variables to save 
        if save_datasets:
            dataset = {"generated_statement_ids": [], "generated_statement_texts": [], "extracted_statement_ids": [], "extracted_statement_texts": [], "prompt_len": []}

        statements = list(statements)

        for i in tqdm(range(0, len(statements), batch_size)):
            # Release GPU memory every 5 batches
            torch.cuda.empty_cache()
            statements_temp = statements[i: i+batch_size]
            if with_chat_template:
                messages = [
                    [
                        {"role": "user", "content": f"{s}\n\n{COT_INSTRUCTIONS}"},
                        {"role": "assistant", "content": "<think>\n"},  # prefill: force continuation inside <think>
                    ]
                    for s in statements_temp
                ]
                inputs = tokenizer.apply_chat_template(
                    messages,
                    add_generation_prompt=False,
                    continue_final_message=True,  # don't close/re-open a turn after our prefill
                    tokenize=True,
                    return_dict=True,
                    return_tensors="pt",
                    padding=True,
                ).to(model.device)
            else:
                prompts = [
                    f"{s}\n\n{COT_INSTRUCTIONS}\n\n<think>\n"
                    for s in statements_temp
                ]
                inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

            with torch.no_grad():
                gen_ids = model.generate(**inputs, max_new_tokens=1000, do_sample=False,
                                        repetition_penalty=1.1,
                                        stop_strings=["Answer: Yes", "Answer: No"],
                                        tokenizer=tokenizer)

            # generate() only returns token IDs, not hidden states, and the batch is padded
            # (left-padding on the prompt, right-padding on generation for rows that finished
            # early) so a shared `[:, -1, :]` index would often land on a pad token instead of
            # real content. For each row: isolate its real (unpadded) generated continuation,
            # find the automated pre-verdict cut point (find_pre_verdict_cut, anchored
            # backward from </think>), then do a separate forward pass over (real prompt +
            # reasoning up to that cut) to get a clean activation that predates the verdict.
            for row_idx, row in enumerate(gen_ids):
                prompt_len = int((inputs["input_ids"][row_idx] != tokenizer.pad_token_id).sum())

                real_ids_full = row[row != tokenizer.pad_token_id].unsqueeze(0)
                gen_part = real_ids_full[0, prompt_len:]
                gen_part_list = gen_part.tolist()
                cut = find_pre_verdict_cut(tokenizer, gen_part_list)

                if verbose:
                    full_text = tokenizer.decode(gen_part_list, skip_special_tokens=True)
                    extracted_text = tokenizer.decode(gen_part_list[:cut], skip_special_tokens=True)
                    print(full_text)
                    print("===== EXTRACTED (this is what the activation is taken from) =====")
                    print(extracted_text)
                    print("---------------------------")

                real_ids = real_ids_full[:, :prompt_len + cut]
                # with torch.no_grad():
                #     fwd_out = model(real_ids, output_hidden_states=True)
                # final_token_activations.append(fwd_out.hidden_states[layer][:, -1, :].cpu())

                if save_datasets:
                    dataset["generated_statement_ids"].append(real_ids_full.cpu().tolist()[0])
                    dataset["generated_statement_texts"].append(tokenizer.decode(real_ids_full[0], skip_special_tokens=True))
                    dataset["extracted_statement_ids"].append(real_ids.cpu().tolist()[0])
                    dataset["extracted_statement_texts"].append(tokenizer.decode(real_ids[0], skip_special_tokens=True))
                    dataset["prompt_len"].append(prompt_len)

        if save_datasets: 
            pd.DataFrame(dataset).to_csv(f"../CoT_datasets/raw/{task}.csv", index=False)

    else:
        dataset = pd.read_csv(dataset_path)
        dataset["generated_statement_ids"] = dataset["generated_statement_ids"].apply(literal_eval).apply(torch.tensor)
        for row_idx, row in enumerate(dataset["generated_statement_ids"]):
            real_ids_full = row[row != tokenizer.pad_token_id].unsqueeze(0)
            gen_part = real_ids_full[0, dataset["prompt_len"][row_idx]:]
            gen_part_list = gen_part.tolist()
            cut = find_pre_verdict_cut(tokenizer, gen_part_list)

            if verbose:
                full_text = tokenizer.decode(gen_part_list, skip_special_tokens=True)
                extracted_text = tokenizer.decode(gen_part_list[:cut], skip_special_tokens=True)
                print(full_text)
                print("===== EXTRACTED (this is what the activation is taken from) =====")
                print(extracted_text)
                print("---------------------------")

            real_ids = real_ids_full[:, :dataset["prompt_len"][row_idx] + cut]
            with torch.no_grad():
                fwd_out = model(real_ids, output_hidden_states=True)
            final_token_activations.append(fwd_out.hidden_states[layer][:, -1, :].cpu())


    # return torch.cat(final_token_activations, dim=0)
    return None

In [ ]:
for task, dataset in [("F5_train", F5_train), ("F5_test", F5_test)]:
    activations = generate_activations_cot(model, dataset["statement"], 16, task, with_chat_template=True, batch_size=12, save_datasets=True, verbose=False)
    dataset["activations_cot_chat"] = list(activations)


100%|██████████| 50/50 [46:11<00:00, 55.43s/it]


In [6]:
for task, dataset in [("A1_train", A1_train), ("A1_test", A1_test)]:
    activations = generate_activations_cot(model, dataset["statement"], 16, task, with_chat_template=True, batch_size=4, save_datasets=True, verbose=False)
    # dataset["activations_cot_chat"] = list(activations)

100%|██████████| 75/75 [30:44<00:00, 24.59s/it]


In [6]:
for task, dataset in [("F2_train", F2_train), ("F2_test", F2_test)]:
    activations = generate_activations_cot(model, dataset["statement"], 16, task, with_chat_template=True, batch_size=10, save_datasets=True, verbose=False)
    # dataset["activations_cot_chat"] = list(activations)

100%|██████████| 52/52 [17:14<00:00, 19.90s/it]


In [6]:
for task, dataset in [("F0_train", F0_train), ("F0_test", F0_test)]:
    activations = generate_activations_cot(model, dataset["statement"], 16, task, with_chat_template=True, batch_size=10, save_datasets=True, verbose=False)
    # dataset["activations_cot_chat"] = list(activations)

100%|██████████| 52/52 [19:17<00:00, 22.25s/it]


In [ ]:
def generate_activations_cot(model, statements, layer, task, with_chat_template=True, batch_size=16, verbose=False, save_datasets=False):
    final_token_activations = []
    dataset_path = Path(f"../CoT_datasets/raw/{task}_layer_{layer}.pkl")
    if not dataset_path.is_file():
        # Define variables to save 
        if save_datasets:
            vectors = []
            dataset = {"generated_statement_ids": [], "generated_statement_texts": [], "extracted_statement_ids": [], "extracted_statement_texts": []}

        statements = list(statements)
        fewshot_pairs = FEWSHOT_DEMOS[task.split("_")[0]]  # [(true-labeled statement, response), (false-labeled statement, response)]

        for i in tqdm(range(0, len(statements), batch_size)):
            statements_temp = statements[i: i+batch_size]
            if with_chat_template:
                messages = [
                    [
                        *[
                            msg
                            for demo_statement, demo_response in fewshot_pairs
                            for msg in (
                                {"role": "user", "content": f"{demo_statement}\n\n{COT_INSTRUCTIONS}"},
                                {"role": "assistant", "content": demo_response},
                            )
                        ],
                        {"role": "user", "content": f"{s}\n\n{COT_INSTRUCTIONS}"},
                        {"role": "assistant", "content": "<think>\n"},  # prefill: force continuation inside <think>
                    ]
                    for s in statements_temp
                ]
                inputs = tokenizer.apply_chat_template(
                    messages,
                    add_generation_prompt=False,
                    continue_final_message=True,  # don't close/re-open a turn after our prefill
                    tokenize=True,
                    return_dict=True,
                    return_tensors="pt",
                    padding=True,
                ).to(model.device)
            else:
                demo_block = "\n\n".join(
                    f"{demo_statement}\n\n{COT_INSTRUCTIONS}\n\n{demo_response}"
                    for demo_statement, demo_response in fewshot_pairs
                )
                prompts = [
                    f"{demo_block}\n\n{s}\n\n{COT_INSTRUCTIONS}\n\n<think>\n"
                    for s in statements_temp
                ]
                inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

            with torch.no_grad():
                gen_ids = model.generate(**inputs, max_new_tokens=1000, do_sample=False,
                                        repetition_penalty=1.1,
                                        stop_strings=["Answer: Yes", "Answer: No"],
                                        tokenizer=tokenizer)

            # generate() only returns token IDs, not hidden states, and the batch is padded
            # (left-padding on the prompt, right-padding on generation for rows that finished
            # early) so a shared `[:, -1, :]` index would often land on a pad token instead of
            # real content. For each row: isolate its real (unpadded) generated continuation,
            # find the automated pre-verdict cut point (find_pre_verdict_cut, anchored
            # backward from </think>), then do a separate forward pass over (real prompt +
            # reasoning up to that cut) to get a clean activation that predates the verdict.
            for row_idx, row in enumerate(gen_ids):
                prompt_len = int((inputs["input_ids"][row_idx] != tokenizer.pad_token_id).sum())

                real_ids_full = row[row != tokenizer.pad_token_id].unsqueeze(0)
                gen_part = real_ids_full[0, prompt_len:]
                gen_part_list = gen_part.tolist()
                cut = find_pre_verdict_cut(tokenizer, gen_part_list)

                if verbose:
                    full_text = tokenizer.decode(gen_part_list, skip_special_tokens=True)
                    extracted_text = tokenizer.decode(gen_part_list[:cut], skip_special_tokens=True)
                    print(full_text)
                    print("===== EXTRACTED (this is what the activation is taken from) =====")
                    print(extracted_text)
                    print("---------------------------")

                real_ids = real_ids_full[:, :prompt_len + cut]
                with torch.no_grad():
                    fwd_out = model(real_ids, output_hidden_states=True)
                final_token_activations.append(fwd_out.hidden_states[layer][:, -1, :].cpu())

                if save_datasets:
                    dataset["generated_statement_ids"].append(real_ids_full.cpu().tolist()[0])
                    dataset["generated_statement_texts"].append(tokenizer.decode(real_ids_full[0], skip_special_tokens=True))
                    dataset["extracted_statement_ids"].append(real_ids.cpu().tolist()[0])
                    dataset["extracted_statement_texts"].append(tokenizer.decode(real_ids[0], skip_special_tokens=True))
                    vectors.append(fwd_out.hidden_states[layer][:, -1, :].cpu())

        if save_datasets: 
            pd.DataFrame(dataset).to_csv(f"../CoT_datasets/raw/{task}.csv", index=False)
            torch.save(torch.cat(vectors), dataset_path)

    else:
        return torch.load(dataset_path)

    return torch.cat(final_token_activations, dim=0)

In [ ]:
def audit_cuts(generations, context_len=40):
    for gen_part in generations:
        cut = find_pre_verdict_cut(tokenizer, gen_part)
        left_text = tokenizer.decode(gen_part[:cut], skip_special_tokens=True)
        right_text = tokenizer.decode(gen_part[cut:], skip_special_tokens=True)

        print(f"...{left_text[-context_len:]}")
        print("===== EXTRACTED (this is what the activation is taken from) =====")
        print(f"{right_text[:context_len]}...")
        print("---------------------------")

def return_generations(model, statements, task, with_chat_template=True, batch_size=16, verbose=False):
    generations = []
    statements = list(statements)
    fewshot_pairs = FEWSHOT_DEMOS[task.split("_")[0]]  # [(true-labeled statement, response), (false-labeled statement, response)]

    for i in range(0, len(statements), batch_size):
        statements_temp = statements[i: i+batch_size]
        if with_chat_template:
            messages = [
                [
                    *[
                        msg
                        for demo_statement, demo_response in fewshot_pairs
                        for msg in (
                            {"role": "user", "content": f"{demo_statement}\n\n{COT_INSTRUCTIONS}"},
                            {"role": "assistant", "content": demo_response},
                        )
                    ],
                    {"role": "user", "content": f"{s}\n\n{COT_INSTRUCTIONS}"},
                    {"role": "assistant", "content": "<think>\n"},  # prefill: force continuation inside <think>
                ]
                for s in statements_temp
            ]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=False,
                continue_final_message=True,  # don't close/re-open a turn after our prefill
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to(model.device)
        else:
            demo_block = "\n\n".join(
                f"{demo_statement}\n\n{COT_INSTRUCTIONS}\n\n{demo_response}"
                for demo_statement, demo_response in fewshot_pairs
            )
            prompts = [
                f"{demo_block}\n\n{s}\n\n{COT_INSTRUCTIONS}\n\n<think>\n"
                for s in statements_temp
            ]
            inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            gen_ids = model.generate(**inputs, max_new_tokens=1000, do_sample=False,
                                     repetition_penalty=1.1,
                                     stop_strings=["Answer: Yes", "Answer: No"],
                                     tokenizer=tokenizer)

        # generate() only returns token IDs, not hidden states, and the batch is padded
        # (left-padding on the prompt, right-padding on generation for rows that finished
        # early) so a shared `[:, -1, :]` index would often land on a pad token instead of
        # real content. For each row: isolate its real (unpadded) generated continuation,
        # find the automated pre-verdict cut point (find_pre_verdict_cut, anchored
        # backward from </think>), then do a separate forward pass over (real prompt +
        # reasoning up to that cut) to get a clean activation that predates the verdict.
        for row_idx, row in enumerate(gen_ids):
            prompt_len = int((inputs["input_ids"][row_idx] != tokenizer.pad_token_id).sum())
            
            real_ids_full = row[row != tokenizer.pad_token_id].unsqueeze(0)
            gen_part = real_ids_full[0, prompt_len:]
            gen_part_list = gen_part.tolist()
            generations.append(gen_part_list)
    return generations

In [ ]:
generations = return_generations(model, F5_train["statement"][:200], 4, "F5", with_chat_template=True, batch_size=16, verbose=True)
audit_cuts(generations, context_len=250)


...0 cities are in Jamaica and exactly 0 are in Haiti. However, according to my understanding, there is 1 city (Port-au-Prince) in Haiti. This contradicts the statement's claim of 0 cities in both Jamaica and Haiti.

Therefore, the statement seems to be
===== EXTRACTED (this is what the activation is taken from) =====
 incorrect because it states that no cities are in either Jamaica or Haiti when in fact, Port-au-Prince is in Haiti.
</think>

The statement claims that none of the listed cities are in Jamaica or Haiti. Upon examining each city:

- **Jamaica**: None...
---------------------------
... two cities, Kisangani and Ribeirao Preto, are in neither Syria nor Eritrea.

So according to my understanding, the statement says exactly 3 are in Syria and 1 in Eritrea. Based on my research, that matches exactly. Therefore, the statement should be
===== EXTRACTED (this is what the activation is taken from) =====
 true.
</think>

The statement claims that exactly 3 cities are in Syria and 1

In [5]:
for task, dataset in [("F5_train", F5_train), ("F5_test", F5_test)]:
    activations = generate_activations_cot(model, dataset["statement"], 16, task, with_chat_template=True, batch_size=16, save_datasets=True)
    dataset["activations_cot_chat"] = list(activations)
    # activations = generate_activations(model, dataset["statement"], 16, task, with_chat_template=False, batch_size=16)
    # dataset["activations"] = list(activations)

100%|██████████| 38/38 [46:00<00:00, 72.65s/it]


In [7]:
for task, dataset in [("A3_train", A3_train), ("A3_test", A3_test)]:
    activations = generate_activations_cot(model, dataset["statement"][:5], 16, task, with_chat_template=True, batch_size=16, save_datasets=False, verbose=True)
    # dataset["activations_cot_chat"] = list(activations)

  0%|          | 0/1 [00:00<?, ?it/s]

Okay, so I have this equation here: (33 + 31) multiplied by (7 times 16), and it's supposed to equal 7171. Hmm, let me break this down step by step to see if that's really true.

First, I'll look at the left side of the equation: (33 + 31). Adding those two numbers together should give me... let's see, 33 plus 30 would be 63, and then adding the remaining 1 makes it 64. So, 33 + 31 equals 64. Got that part.

Next up is the right side inside the parentheses: 7 multiplied by 16. I know that 7 times 10 is 70, and 7 times 6 is 42. If I add those together, 70 plus 42 gives me 112. So, 7 times 16 is 112. That seems right.

Now, the original expression becomes 64 multiplied by 112. Let me calculate that. Maybe breaking it down will help. 64 times 100 is 6400, and 64 times 12 is... well, 64 times 10 is 640, and 64 times 2 is 128. Adding those together, 640 plus 128 is 768. So, 64 times 112 is 6400 plus 768, which equals 7168.

Wait a minute, but the problem says it's equal to 7171. That doesn'

100%|██████████| 1/1 [00:27<00:00, 27.34s/it]


Okay, so I have this equation here: (27 * 23) * (19 + 1) equals 12420. I need to figure out if that's true or not. Let me break it down step by step.

First, let's look at the left side of the equation. There are two main parts multiplied together: (27 * 23) and (19 + 1). I should calculate each part separately and then multiply them to see if they equal 12420.

Starting with the first part, 27 multiplied by 23. Hmm, I'm a bit rusty on my multiplication, but I can work it out. Let me do 27 times 20 first because that's easier. 27 * 20 is 540. Then, I need to add 27 more because 23 is 20 plus 3. So, 540 plus 27 is... let me add that up. 540 + 20 is 560, and then +7 makes 567. Wait, no, that doesn't seem right. Let me try another way. Maybe using the standard multiplication method:

  27  
x23  
------  
  81  (which is 27*3)  
870   (which is 27*20, shifted one position to the left)  
------  
Adding those together: 81 + 870. 81 + 800 is 881, plus 70 is 951. Oh, okay, so 27*23 is 621. W

  0%|          | 0/1 [00:00<?, ?it/s]

Okay, so I have this equation here: (31 * 18) + (5 - 6) = 554. I need to figure out if this statement is true or false by breaking down each part step by step. Let me start by looking at each component separately.

First, let's tackle the multiplication part: 31 multiplied by 18. I remember that multiplying two numbers can be done using the distributive property or just straight multiplication. Let me do it step by step. 

So, 31 times 18. I'll break it down as 30*18 plus 1*18 because 31 is 30+1. Calculating 30*18 first: 30 times 10 is 300, and 30 times 8 is 240, so adding those together gives 540. Then, 1*18 is just 18. Adding those results together: 540 + 18 equals 558. So, 31*18 equals 558.

Next, I'll look at the subtraction part: 5 minus 6. That seems straightforward. Subtracting a larger number from a smaller one gives a negative result. So, 5 - 6 should be -1.

Now, putting it all back into the original equation: we've got 558 from the multiplication and -1 from the subtraction.

100%|██████████| 1/1 [00:25<00:00, 25.46s/it]


### Training the Model and Extracting AUROC Scores

In [6]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def train_probe_pytorch(activations, labels, device='cuda'):
    (X_train, X_test), (y_train, y_test) = activations, labels

    X_train = torch.stack(list(X_train)).float().numpy()
    X_test = torch.stack(list(X_test)).float().numpy()
    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()


    # Mean-center using only the training mean
    train_mean = X_train.mean(axis=0)
    X_train = X_train - train_mean
    X_test = X_test - train_mean

    # Convert to tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

    hidden_dim = X_train.shape[1]

    # THIS is the entire "model": one linear layer, no bias.
    # w(x) = w^T x, no offset term -> passes through the origin.
    probe = nn.Linear(hidden_dim, 1, bias=False).to(device)

    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()  # sigmoid + binary cross-entropy, combined for numerical stability

    for step in range(1000):
        optimizer.zero_grad()
        logits = probe(X_train_t).squeeze(-1)   # w^T x for every example
        loss = loss_fn(logits, y_train_t)
        loss.backward()
        optimizer.step()

    # Evaluate
    probe.eval()
    with torch.no_grad():
        test_logits = probe(X_test_t).squeeze(-1).cpu().numpy()
    auroc = roc_auc_score(y_test, test_logits)

    return probe, train_mean, auroc

In [8]:
def get_AUROC_score(generate_activations, generate_activations_kwargs: dict, task_name: str, train_dataset: pd.DataFrame, test_dataset: pd.DataFrame):
    activations_train = generate_activations(**generate_activations_kwargs, statements=train_dataset["statement"])
    activations_test = generate_activations(**generate_activations_kwargs, statements=test_dataset["statement"])

    act_kw = "activations_chat" if generate_activations_kwargs["with_chat_template"] else "activations"

    train_dataset[act_kw] = list(activations_train)
    test_dataset[act_kw] = list(activations_test)

    probe, _, auroc = train_probe_pytorch(
        (train_dataset[act_kw], test_dataset[act_kw]),
        (train_dataset["label"], test_dataset["label"])
    )

    print(f"The AUROC score for {task_name} at layer {generate_activations_kwargs['layer']} is {auroc}.")


### Baseline AUROC Scores (Without CoT, just the plain statements)

In [9]:
for task_name, train_set, test_set in tqdm([["F0", F0_train, F0_test],
                                            ["F1", F1_train, F1_test],
                                            ["F2", F2_train, F2_test],
                                            ["F3", F3_train, F3_test],
                                            ["F4", F4_train, F4_test],
                                            ["F5", F5_train, F5_test], 
                                            ["A1", A1_train, A1_test],
                                            ["A2", A2_train, A2_test], 
                                            ["A3", A3_train, A3_test]]):
    get_AUROC_score(generate_activations=generate_activations,
                    generate_activations_kwargs={
                        "layer": 16,
                        "with_chat_template": True,
                        "batch_size": 16,
                        "model": model
                    },
                    task_name=task_name,
                    train_dataset=train_set,
                    test_dataset=test_set)

 11%|█         | 1/9 [00:06<00:50,  6.35s/it]

The AUROC score for F0 at layer 16 is 0.9941253662109375.


 22%|██▏       | 2/9 [00:12<00:44,  6.32s/it]

The AUROC score for F1 at layer 16 is 0.995758056640625.


 33%|███▎      | 3/9 [00:24<00:52,  8.83s/it]

...

The AUROC score for A1 at layer 16 is 0.5652444444444444.


 89%|████████▉ | 8/9 [02:09<00:15, 15.48s/it]

The AUROC score for A2 at layer 16 is 0.5357777777777778.


100%|██████████| 9/9 [02:13<00:00, 14.79s/it]

The AUROC score for A3 at layer 16 is 0.5266222222222222.


In [7]:
train_probe_pytorch((F5_train["activations_cot_chat"], F5_test["activations_cot_chat"]), (F5_train["label"], F5_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.1769136 , -0.02237735, -0.01662826, ...,  0.04190185,
         0.09452999, -0.06981217], shape=(4096,), dtype=float32),
 0.9643848393848394)

In [15]:
train_probe_pytorch((F5_train["activations_chat_deafult"], F5_test["activations_chat_deafult"]), (F5_train["label"], F5_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.14653274,  0.13728258, -0.03395567, ..., -0.04017596,
        -0.0878389 ,  0.02431238], shape=(4096,), dtype=float32),
 0.7964669214669214)